# 02_jugadores
Baja las 6 llamadas de la API para cada jugador en cada ciclo semanal: los nuevos y los activos (los retirados quedan congelados).
Un ciclo se cierra cuando todo jugador tiene sus 6 llamadas bajadas desde que abrió el ciclo, o agotó los intentos. Recién ahí corren silver y gold.
Se puede cortar y retomar en cualquier momento: lo pendiente se calcula siempre desde `bronze_player_raw`.
Si Transfermarkt frena (202/403/429) deja una pausa en `bronze_corridas` y las corridas siguientes salen sin hacer pedidos hasta que pase.

Es la fase 4 del pipeline (datos por jugador), la que `01_extraccion` deja afuera.

## Contenido

| Celda | Qué hace |
|---|---|
| `%run ./ciclo` | Carga el estado del ciclo semanal |
| Configuración | Widgets, nombres de tablas, esquemas y `registrar` |
| 0) Ciclo | Sale si hoy no hay que bajar jugadores |
| 1) Pausa y tope diario | Sale si hay una pausa vigente o si ya se usó el tope de horas |
| 2) Qué falta en este ciclo | Calcula las llamadas pendientes sin levantar la API |
| 3) Levantar la API | Descarga y arranca la API local |
| 4) Bucle principal | Hace las llamadas, escribe en `bronze_player_raw` y cierra el ciclo |

## Tablas

| Tabla | Uso |
|---|---|
| `bronze_player_raw` | Lee lo ya bajado y las fechas de los errores; escribe cada llamada con `append` |
| `bronze_corridas` | Lee pausas y horas usadas; escribe una fila por corrida |
| `bronze_ciclos` | Estado del ciclo (a través de `ciclo`); marca `cerrado_en` |
| `bronze_departures`, `bronze_arrivals`, `bronze_universo_extra` | Lee los jugadores del universo |
| `bronze_fuera_alcance` | Lee los jugadores que quedan excluidos. La crea vacía si no existe |

## Las 6 llamadas

`LLAMADAS`, en orden de prioridad: `transfers`, `stats`, `profile`, `market_value`, `national_career` y `achievements`.

## Estados en `bronze_player_raw`

| Estado | Cuándo |
|---|---|
| `hecho` | HTTP 200 con JSON válido; se guarda el cuerpo completo |
| `sin_datos` | HTTP 404; se guardan los primeros 300 caracteres de la respuesta |
| `error` | Cualquier otro caso, incluido un HTTP 200 que no es JSON válido; se guardan los primeros 300 caracteres |

## Carga de `ciclo`
`%run ./ciclo` define `ahora`, `ciclo_vigente`, `marcar` y `motivo_no_corre` (ver el notebook `ciclo`).

In [ ]:
%run ./ciclo

## Configuración

| Widget | Valor por defecto | Para qué sirve |
|---|---|---|
| `ESPERA_API` | `1.0` | Segundos de espera entre llamadas |
| `MAX_HORAS` | `24` | Tope de horas de esta corrida |
| `MAX_HORAS_DIA` | `24` | Tope de horas usadas en las últimas 24 h (suma de `bronze_corridas`). Con `24` no hay tope |
| `MAX_INTENTOS` | `3` | Errores por (jugador, llamada) y por ciclo antes de dejarlo |
| `MAX_ERRORES_SEGUIDOS` | `5` | Errores consecutivos que cortan la corrida |
| `VOLCAR_CADA` | `20` | Llamadas entre cada escritura a Delta |
| `PAUSA_HORAS` | `6` | Duración de la pausa tras un freno de Transfermarkt |
| `API_REF` | commit de `felipeall/transfermarkt-api` | Versión de la API que se descarga |

La celda también define los nombres de las tablas (`RAW`, `CORR`), los esquemas, crea `bronze_player_raw`, `bronze_corridas` y `bronze_fuera_alcance` (vacía) si no existen y define `registrar(motivo, llamadas, errores, pausa)`, que agrega una fila a `bronze_corridas` con el inicio de esta corrida.

In [ ]:
import json, os, sys, time, subprocess, urllib.request, urllib.error, zipfile
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

dbutils.widgets.text("ESPERA_API", "1.0")             # segundos entre llamadas
dbutils.widgets.text("MAX_HORAS", "24")                # tope de esta corrida
dbutils.widgets.text("MAX_HORAS_DIA", "24")           # tope de horas en las ultimas 24 h (suma de bronze_corridas); 24 = sin tope, para medir la cuota
dbutils.widgets.text("MAX_INTENTOS", "3")             # errores por (jugador, llamada) y por ciclo antes de dejarlo
dbutils.widgets.text("MAX_ERRORES_SEGUIDOS", "5")
dbutils.widgets.text("VOLCAR_CADA", "20")             # llamadas entre cada escritura a Delta
dbutils.widgets.text("PAUSA_HORAS", "6")              # pausa tras un freno de Transfermarkt
dbutils.widgets.text("API_REF", "71aa75ecd708e5c8c283c4af5968dd370777b3f8")

ESPERA_API = float(dbutils.widgets.get("ESPERA_API"))
MAX_HORAS = float(dbutils.widgets.get("MAX_HORAS"))
MAX_HORAS_DIA = float(dbutils.widgets.get("MAX_HORAS_DIA"))
MAX_INTENTOS = int(dbutils.widgets.get("MAX_INTENTOS"))
MAX_ERRORES_SEGUIDOS = int(dbutils.widgets.get("MAX_ERRORES_SEGUIDOS"))
VOLCAR_CADA = int(dbutils.widgets.get("VOLCAR_CADA"))
PAUSA_HORAS = float(dbutils.widgets.get("PAUSA_HORAS"))
API_REF = dbutils.widgets.get("API_REF")

LLAMADAS = ["transfers", "stats", "profile", "market_value", "national_career", "achievements"]  # orden de prioridad

API = "http://127.0.0.1:8000"
RAW = "workspace.inferiores.bronze_player_raw"
CORR = "workspace.inferiores.bronze_corridas"

SCHEMA_RAW = StructType([
    StructField("player_id", StringType()), StructField("llamada", StringType()),
    StructField("estado", StringType()), StructField("http", IntegerType()),
    StructField("body", StringType()), StructField("fecha", StringType()),
])
SCHEMA_CORR = StructType([
    StructField("inicio", StringType()), StructField("fin", StringType()),
    StructField("llamadas", IntegerType()), StructField("errores", IntegerType()),
    StructField("motivo", StringType()), StructField("pausa_hasta", StringType()),
])

INICIO = ahora()
T0 = time.time()

spark.sql(f"""CREATE TABLE IF NOT EXISTS {RAW}
              (player_id STRING, llamada STRING, estado STRING, http INT, body STRING, fecha STRING)
              USING DELTA""")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CORR}
              (inicio STRING, fin STRING, llamadas INT, errores INT, motivo STRING, pausa_hasta STRING)
              USING DELTA""")

# Jugadores fuera del alcance (se excluyen de los pendientes). Si no existe, se crea vacia.
spark.sql("""CREATE TABLE IF NOT EXISTS workspace.inferiores.bronze_fuera_alcance
              (player_id STRING, motivo STRING, detectado_en STRING)
              USING DELTA""")

def registrar(motivo, llamadas=0, errores=0, pausa=None):
    spark.createDataFrame([(INICIO, ahora(), llamadas, errores, motivo, pausa)], SCHEMA_CORR) \
         .write.mode("append").saveAsTable(CORR)

## 0) Ciclo: ¿hay que bajar jugadores hoy?
`motivo_no_corre("jugadores")` devuelve por qué no corresponde correr (falta terminar la extracción, o los jugadores del ciclo ya están completos). En ese caso se registra `saltea: <motivo>` en `bronze_corridas` y el notebook termina con `SALTEADO: <motivo>`. Si corresponde, imprime el ciclo vigente.

In [ ]:
# 0) Ciclo: hay que bajar jugadores hoy?
motivo_salida = motivo_no_corre("jugadores")
if motivo_salida:
    registrar("saltea: " + motivo_salida)
    dbutils.notebook.exit("SALTEADO: " + motivo_salida)
c_ciclo = ciclo_vigente()
print(f"ciclo {c_ciclo['lunes']} | abierto {c_ciclo['inicio']}")

## 1) Pausa por freno y tope de horas por día
- **Pausa:** si la última `pausa_hasta` de `bronze_corridas` todavía no venció, termina con `PAUSA hasta <fecha>`.
- **Tope diario (1b):** suma las horas de las corridas que empezaron en las últimas 24 h (ventana móvil, porque no se sabe a qué hora se reinicia la cuota del cómputo). Si la suma alcanza `MAX_HORAS_DIA`, termina con `TOPE DIARIO`. Si no, el límite de esta corrida es el menor entre `MAX_HORAS` y las horas que quedan del tope (`LIMITE_HORAS`).

In [ ]:
# 1) Pausa por freno de Transfermarkt: si sigue vigente, salir sin hacer nada
pausa_vigente = spark.sql(f"SELECT MAX(pausa_hasta) AS p FROM {CORR}").first().p
if pausa_vigente and pausa_vigente > ahora():
    registrar("pausa activa", pausa=pausa_vigente)
    dbutils.notebook.exit(f"PAUSA hasta {pausa_vigente}")

# 1b) Tope de horas por dia: suma de las corridas que empezaron en las ultimas 24 h (ventana movil,
#     porque no se sabe a que hora se reinicia la cuota). Protege contra agotar la cuota todos los dias.
desde_24h = (datetime.now() - timedelta(hours=24)).isoformat(timespec="seconds")
usadas = spark.sql(f"""SELECT COALESCE(SUM((unix_timestamp(to_timestamp(fin)) - unix_timestamp(to_timestamp(inicio))) / 3600), 0) AS h
                       FROM {CORR} WHERE inicio >= '{desde_24h}'""").first().h
if usadas >= MAX_HORAS_DIA:
    registrar(f"tope diario: {usadas:.1f} h usadas en las ultimas 24 h")
    dbutils.notebook.exit(f"TOPE DIARIO ({usadas:.1f} h de {MAX_HORAS_DIA} h)")
LIMITE_HORAS = min(MAX_HORAS, MAX_HORAS_DIA - usadas)
print(f"horas usadas en 24 h: {usadas:.2f} | limite de esta corrida: {LIMITE_HORAS:.2f} h")

## 2) Qué falta en este ciclo
Se calcula solo con Spark, sin levantar la API. `calcular_pendientes()` devuelve las llamadas nuevas, las de refresco, la cantidad de jugadores y la de activos.

| Concepto | Cómo se arma |
|---|---|
| Jugadores (`orden`) | Los de `bronze_departures`, `bronze_arrivals` y `bronze_universo_extra`, sin los de `bronze_fuera_alcance`, ordenados por la última temporada en que aparecen (más reciente primero) |
| Llamadas hechas (`ok`) | Para cada (jugador, llamada), la fecha más reciente con estado `hecho` o `sin_datos` |
| Errores del ciclo (`err`) | Errores con fecha desde el inicio del ciclo y posteriores al último dato bueno, contados por (jugador, llamada) |
| Activos | Jugadores cuyo último `profile` bajado tiene `isRetired` en `false`. Los demás quedan congelados |
| Nuevas | (llamada, jugador) sin dato bueno y con menos de `MAX_INTENTOS` errores |
| Refresco | (llamada, jugador activo) con dato bueno anterior al inicio del ciclo y con menos de `MAX_INTENTOS` errores |

Un par que agotó los intentos cuenta como cerrado. Las tareas se ordenan primero las nuevas y después las de refresco, y dentro de cada grupo por llamada según el orden de `LLAMADAS`. Imprime la cantidad de nuevas y de refresco por llamada.

Si no queda nada pendiente, marca `cerrado_en` en `bronze_ciclos`, registra `sin pendientes: ciclo cerrado` y termina con `CICLO CERRADO`.

In [ ]:
# 2) Que falta en este ciclo (solo Spark, sin levantar la API)
CICLO_INI = c_ciclo["inicio"]

def calcular_pendientes():
    orden = [r.player_id for r in spark.sql("""
        SELECT CAST(player_id AS STRING) AS player_id, MAX(season) AS ult FROM (
            SELECT player_id, season FROM workspace.inferiores.bronze_departures
            UNION ALL
            SELECT player_id, season FROM workspace.inferiores.bronze_arrivals
            UNION ALL
            SELECT player_id, season FROM workspace.inferiores.bronze_universo_extra)
        WHERE CAST(player_id AS STRING) NOT IN
              (SELECT player_id FROM workspace.inferiores.bronze_fuera_alcance)
        GROUP BY 1 ORDER BY ult DESC, player_id""").collect()]

    ok = {}
    for r in spark.sql(f"""SELECT player_id, llamada, MAX(fecha) AS f FROM {RAW}
                           WHERE estado IN ('hecho','sin_datos') GROUP BY 1,2""").collect():
        ok[(r.player_id, r.llamada)] = r.f

    err = {}   # errores de ESTE ciclo, posteriores al ultimo dato bueno
    for r in spark.sql(f"""SELECT player_id, llamada, fecha FROM {RAW}
                           WHERE estado = 'error' AND fecha >= '{CICLO_INI}'""").collect():
        k = (r.player_id, r.llamada)
        if k not in ok or r.fecha > ok[k]:
            err[k] = err.get(k, 0) + 1

    # jugadores activos: ultimo profile bajado con isRetired = false
    activos = {r.player_id for r in spark.sql(f"""
        SELECT player_id FROM (
            SELECT player_id, get_json_object(body, '$.isRetired') AS retirado,
                   ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY fecha DESC) AS rn
            FROM {RAW} WHERE llamada = 'profile' AND estado = 'hecho')
        WHERE rn = 1 AND retirado = 'false'""").collect()}

    nuevos, refresco = [], []
    for ll in LLAMADAS:
        for pid in orden:
            k = (pid, ll)
            if err.get(k, 0) >= MAX_INTENTOS:
                continue                      # agoto los intentos: cuenta como cerrado
            if k not in ok:
                nuevos.append((ll, pid, "nuevo"))
            elif pid in activos and ok[k] < CICLO_INI:
                refresco.append((ll, pid, "refresco"))
    return nuevos, refresco, len(orden), len(activos)

nuevos, refresco, n_jug, n_act = calcular_pendientes()
tareas = nuevos + refresco

print(f"jugadores: {n_jug} | activos conocidos: {n_act}")
for ll in LLAMADAS:
    n1 = sum(1 for t in nuevos if t[0] == ll)
    n2 = sum(1 for t in refresco if t[0] == ll)
    print(f"{ll}: nuevos {n1} | refresco {n2}")

if not tareas:
    marcar("cerrado_en")
    registrar("sin pendientes: ciclo cerrado")
    dbutils.notebook.exit("CICLO CERRADO")

## 3) Levantar la API
`levantar_api()`: si la API ya responde en `127.0.0.1:8000`, no hace nada. Si no, descarga el código de `felipeall/transfermarkt-api` en el commit `API_REF`, lo instala con `pip` (extra `server`), lo arranca con `uvicorn` y espera hasta 60 s a que responda.

In [ ]:
# 3) Levantar la API (si ya esta arriba, no hace nada)
def levantar_api():
    try:
        urllib.request.urlopen(API + "/openapi.json", timeout=10)
        print("API ya levantada")
        return
    except Exception:
        pass
    url = f"https://github.com/felipeall/transfermarkt-api/archive/{API_REF}.zip"
    urllib.request.urlretrieve(url, "/tmp/tmapi.zip")
    z = zipfile.ZipFile("/tmp/tmapi.zip")
    raiz = "/tmp/" + z.namelist()[0].split("/")[0]
    z.extractall("/tmp")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", raiz + "[server]"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("pip fallo: " + r.stderr[-300:])
    env = dict(os.environ, PYTHONPATH=raiz)
    log = open("/tmp/uvicorn.log", "w")
    subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
                     cwd=raiz, env=env, stdout=log, stderr=log)
    for _ in range(30):
        time.sleep(2)
        try:
            urllib.request.urlopen(API + "/openapi.json", timeout=10)
            print("API levantada")
            return
        except Exception:
            pass
    raise RuntimeError("La API no levanto en 60 s")

levantar_api()

## 4) Bucle principal

| Elemento | Qué hace |
|---|---|
| `Corte(msg, fallo, pausa)` | Excepción que corta la corrida. `fallo` hace que el notebook termine en error; `pausa` registra una pausa |
| `llamar(pid, ll)` | `GET /players/{pid}/{ll}`; devuelve `(código HTTP, cuerpo)`. Devuelve `-1` ante un error de conexión y `-2` ante cualquier otro error |
| `volcar()` | Escribe el búfer en `bronze_player_raw` con `append` |

**Por cada tarea:** controla el tope de horas, espera `ESPERA_API` s, llama a la API y guarda la fila (`player_id`, `llamada`, `estado`, `http`, `body`, `fecha`). Escribe cada `VOLCAR_CADA` llamadas e imprime el avance cada 500.

**Cortes**

| Causa | Termina en error | Pausa |
|---|---|---|
| Tope de horas (`LIMITE_HORAS`) | No | No |
| HTTP `202`, `403` o `429` | Sí | Sí, `PAUSA_HORAS` horas |
| API sin respuesta (`Connection refused`) | Sí | No |
| `MAX_ERRORES_SEGUIDOS` errores seguidos | Sí | No |

Pase lo que pase, al final se escribe lo pendiente del búfer y se registra la corrida en `bronze_corridas` (motivo, llamadas, errores y `pausa_hasta`). Si la corrida recorrió todo lo pendiente (`TERMINADO`), vuelve a calcular los pendientes y, si no queda ninguno, marca `cerrado_en` y agrega `CICLO CERRADO` al motivo. Si hubo un corte con `fallo`, lanza `RuntimeError`; si no, termina con el motivo como texto de salida.

In [ ]:
# 4) Bucle principal
class Corte(Exception):
    def __init__(self, msg, fallo=False, pausa=False):
        super().__init__(msg)
        self.fallo = fallo
        self.pausa = pausa

def llamar(pid, ll):
    try:
        with urllib.request.urlopen(f"{API}/players/{pid}/{ll}", timeout=90) as x:
            return x.status, x.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode("utf-8", "replace")[:300]
    except urllib.error.URLError as e:
        return -1, str(e)[:300]
    except Exception as e:
        return -2, str(e)[:300]

buf = []
def volcar():
    if buf:
        spark.createDataFrame(buf, SCHEMA_RAW).write.mode("append").saveAsTable(RAW)
        buf.clear()

n, n_err, seguidos, lat = 0, 0, 0, 0.0
motivo, fallo, pausa_hasta = "INTERRUMPIDA (excepcion no prevista)", False, None
try:
    for ll, pid, fase in tareas:
        if time.time() - T0 > LIMITE_HORAS * 3600:
            raise Corte(f"tope de {LIMITE_HORAS:.2f} h alcanzado")
        time.sleep(ESPERA_API)
        t1 = time.time()
        code, body = llamar(pid, ll)
        lat += time.time() - t1
        n += 1

        if code in (202, 403, 429):
            raise Corte(f"FRENO: HTTP {code} en {ll} del jugador {pid}", fallo=True, pausa=True)
        if code == -1 and "refused" in body.lower():
            raise Corte("La API no responde (Connection refused)", fallo=True)

        if code == 200:
            try:
                json.loads(body)
                estado = "hecho"
            except Exception:
                estado = "error"
        elif code == 404:
            estado = "sin_datos"
        else:
            estado = "error"

        if estado == "error":
            n_err += 1
            seguidos += 1
        else:
            seguidos = 0
        buf.append((pid, ll, estado, int(code), body if estado == "hecho" else body[:300], ahora()))

        if len(buf) >= VOLCAR_CADA:
            volcar()
        if n % 500 == 0:
            print(f"{n} llamadas | {(time.time()-T0)/3600:.2f} h | latencia media {lat/n:.2f} s | {fase} {ll}", flush=True)
        if seguidos >= MAX_ERRORES_SEGUIDOS:
            raise Corte(f"{MAX_ERRORES_SEGUIDOS} errores seguidos (ultimo: http {code}, {body[:120]})", fallo=True)
    motivo = "TERMINADO"
except Corte as c:
    motivo, fallo = str(c), c.fallo
    if c.pausa:
        pausa_hasta = (datetime.now() + timedelta(hours=PAUSA_HORAS)).isoformat(timespec="seconds")
finally:
    try:
        volcar()
    except Exception as e:
        print("No se pudo volcar lo ultimo:", str(e)[:200])
    try:
        registrar(motivo, n, n_err, pausa_hasta)
    except Exception as e:
        print("No se pudo registrar la corrida:", str(e)[:200])

# Si se recorrio todo lo pendiente, recalcular: sin pendientes (o con intentos agotados) = ciclo cerrado
if motivo == "TERMINADO":
    nuevos2, refresco2, _, _ = calcular_pendientes()
    if not (nuevos2 or refresco2):
        marcar("cerrado_en")
        motivo += " | CICLO CERRADO"

print(f"{motivo} | llamadas: {n} | errores: {n_err} | {(time.time()-T0)/3600:.2f} h")
if fallo:
    raise RuntimeError(motivo)
dbutils.notebook.exit(motivo)